# Jira Theme + Idea Card Audit Notebook

This notebook audits Jira tickets after a chosen date and answers:

1. Which source tickets are in scope after the date?
2. Which of those tickets are **valid** because they link to at least one `GROUP-*` ticket whose `issueType = "Theme"`?
3. For each valid ticket, what value streams are attached?  
   In this notebook, **Theme summaries are treated as the value stream signal**.
4. Does each valid ticket appear to have an **idea card**?
   - **clear** = explicit `idea card` wording found in text or attachment names
   - **implied** = no explicit wording, but likely supporting attachment exists such as PPT/PPTX/PDF/DOC/DOCX
   - **none** = no signal found

The notebook exports both CSV and JSON outputs.

In [ ]:
%pip install -q neo4j pandas

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import json
import re
import ast
from datetime import datetime, timezone
from typing import Any

## 1) Configuration

In [ ]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "YOUR_PASSWORD"
NEO4J_DATABASE = "neo4j"

SINCE_DATE = "2023-01-01"
SOURCE_ISSUE_TYPE = "Engagement Request"   # set to None for all source ticket types
THEME_ISSUE_TYPE = "Theme"

GROUP_LINK_ARRAY_FIELD = "inwardIssues"

PREFERRED_TEXT_KEYS = ["description", "summary", "details"]
ATTACHMENT_KEY_HINTS = ["attach", "file", "document", "ppt", "deck", "slide"]

SINCE_EPOCH = int(datetime.fromisoformat(SINCE_DATE + "T00:00:00+00:00").timestamp())
SINCE_EPOCH

## 2) Neo4j helpers

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)

def run_query(cypher: str, params: dict | None = None):
    with driver.session(database=NEO4J_DATABASE) as session:
        return list(session.run(cypher, params or {}))

print("Connected")

## 3) Quick schema recap / inspection

In [ ]:
schema_freq_query = '''
MATCH (n:JIRA)
WHERE n.creationDateEpoch >= $since_epoch
UNWIND keys(n) AS k
RETURN k AS property_name, count(*) AS freq
ORDER BY freq DESC, property_name
'''

schema_freq_df = pd.DataFrame([r.data() for r in run_query(schema_freq_query, {"since_epoch": SINCE_EPOCH})])
schema_freq_df.head(50)

In [ ]:
SAMPLE_TICKET_KEY = None   # e.g. "IDMT-19761"

if SAMPLE_TICKET_KEY:
    sample_query = '''
    MATCH (n:JIRA {key: $ticket_key})
    RETURN n AS ticket
    '''
    sample_rows = run_query(sample_query, {"ticket_key": SAMPLE_TICKET_KEY})
    if sample_rows:
        sample_ticket = dict(sample_rows[0]["ticket"])
        print("Ticket keys:", sorted(sample_ticket.keys()))
        sample_ticket
    else:
        print("No ticket found for", SAMPLE_TICKET_KEY)
else:
    print("Set SAMPLE_TICKET_KEY if you want to inspect one ticket.")

## 4) Fetch source tickets and linked Theme tickets

Working schema used in this notebook:

- Source ticket: `(:JIRA)` with
  - `creationDateEpoch >= SINCE_EPOCH`
  - optionally `issueType = SOURCE_ISSUE_TYPE`
- Linked group keys are taken from `t.inwardIssues` and filtered to keys starting with `GROUP-`
- Linked group node must be another `(:JIRA)` with `key = group_key`
- A linked group counts as a valid value stream signal if `g.issueType = "Theme"`

In [ ]:
def fetch_source_theme_rows(since_epoch: int, source_issue_type: str | None = None, theme_issue_type: str = "Theme"):
    issue_type_filter = ""
    if source_issue_type:
        issue_type_filter = "AND t.issueType = $source_issue_type"

    cypher = f'''
    MATCH (t:JIRA)
    WHERE t.creationDateEpoch >= $since_epoch
      {issue_type_filter}

    WITH
        t,
        [k IN coalesce(t.{GROUP_LINK_ARRAY_FIELD}, []) WHERE k STARTS WITH "GROUP-"] AS candidate_group_keys

    UNWIND CASE
        WHEN size(candidate_group_keys) = 0 THEN [NULL]
        ELSE candidate_group_keys
    END AS group_key

    OPTIONAL MATCH (g:JIRA {{key: group_key}})
    WHERE g.issueType = $theme_issue_type

    RETURN
        t AS ticket_node,
        group_key AS linked_group_key,
        g AS theme_node
    ORDER BY t.creationDateEpoch DESC, t.key, linked_group_key
    '''

    params = {
        "since_epoch": since_epoch,
        "theme_issue_type": theme_issue_type,
    }
    if source_issue_type:
        params["source_issue_type"] = source_issue_type

    rows = run_query(cypher, params)
    out = []
    for row in rows:
        ticket_node = row["ticket_node"]
        theme_node = row["theme_node"]
        out.append({
            "ticket": dict(ticket_node) if ticket_node is not None else None,
            "linked_group_key": row["linked_group_key"],
            "theme": dict(theme_node) if theme_node is not None else None,
        })
    return out

source_theme_rows = fetch_source_theme_rows(
    since_epoch=SINCE_EPOCH,
    source_issue_type=SOURCE_ISSUE_TYPE,
    theme_issue_type=THEME_ISSUE_TYPE,
)

print("Raw source/theme rows:", len(source_theme_rows))
print("Distinct source tickets:", len({r['ticket']['key'] for r in source_theme_rows if r['ticket']}))

## 5) Helper functions for idea card detection

### `clear`
- the text explicitly contains `idea card`
- or attachment name explicitly contains `idea card`

### `implied`
- no explicit wording, but supporting attachments exist, especially:
  - `.ppt`, `.pptx`, `.pdf`, `.doc`, `.docx`
- or other attachment-like fields contain filenames/URLs

### `none`
- no clear or implied signal found

In [ ]:
URL_RE = re.compile(r"https?://\S+", re.IGNORECASE)
IDEA_CARD_RE = re.compile(r"\bidea\s*card\b", re.IGNORECASE)
PPT_RE = re.compile(r"\.(ppt|pptx)$", re.IGNORECASE)
DOC_RE = re.compile(r"\.(pdf|doc|docx)$", re.IGNORECASE)
LIKELY_DECK_RE = re.compile(r"(ppt|pptx|powerpoint|deck|slides?)", re.IGNORECASE)

def safe_json_or_literal_parse(text: str):
    if not isinstance(text, str):
        return text
    s = text.strip()
    if not s:
        return text
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        for fn in (json.loads, ast.literal_eval):
            try:
                return fn(s)
            except Exception:
                pass
    return text

def flatten_to_strings(value: Any) -> list[str]:
    value = safe_json_or_literal_parse(value)

    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple, set)):
        out = []
        for item in value:
            out.extend(flatten_to_strings(item))
        return out
    if isinstance(value, dict):
        out = []
        for k, v in value.items():
            out.append(str(k))
            out.extend(flatten_to_strings(v))
        return out
    return [str(value)]

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

def collect_text_fields(ticket: dict) -> dict[str, str]:
    text_fields = {}
    for k, v in ticket.items():
        key_l = k.lower()
        if isinstance(v, str) and v.strip():
            text_fields[k] = normalize_whitespace(v)
        elif any(hint in key_l for hint in ["description", "summary", "detail", "comment", "notes", "body"]):
            joined = " | ".join(normalize_whitespace(x) for x in flatten_to_strings(v) if str(x).strip())
            if joined:
                text_fields[k] = joined
    return text_fields

def collect_attachment_like_strings(ticket: dict) -> dict[str, list[str]]:
    out = {}
    for k, v in ticket.items():
        key_l = k.lower()
        if any(hint in key_l for hint in ATTACHMENT_KEY_HINTS):
            strs = [normalize_whitespace(x) for x in flatten_to_strings(v) if str(x).strip()]
            if strs:
                out[k] = strs
    return out

def unique_keep_order(items: list[str]) -> list[str]:
    seen = set()
    out = []
    for item in items:
        item_n = normalize_whitespace(item)
        if item_n and item_n not in seen:
            seen.add(item_n)
            out.append(item_n)
    return out

def classify_idea_card(ticket: dict) -> dict:
    text_fields = collect_text_fields(ticket)
    attachment_fields = collect_attachment_like_strings(ticket)

    explicit_hits = []
    explicit_links = []
    attachment_names = []
    ppt_attachments = []
    doc_attachments = []

    for field_name, field_text in text_fields.items():
        if IDEA_CARD_RE.search(field_text):
            explicit_hits.append(f"text:{field_name}")
            explicit_links.extend(URL_RE.findall(field_text))

    for field_name, values in attachment_fields.items():
        for value in values:
            attachment_names.append(value)

            if IDEA_CARD_RE.search(value):
                explicit_hits.append(f"attachment:{field_name}")

            if PPT_RE.search(value) or LIKELY_DECK_RE.search(value):
                ppt_attachments.append(value)

            if DOC_RE.search(value):
                doc_attachments.append(value)

            explicit_links.extend(URL_RE.findall(value))

    attachment_names = unique_keep_order(attachment_names)
    ppt_attachments = unique_keep_order(ppt_attachments)
    doc_attachments = unique_keep_order(doc_attachments)
    explicit_links = unique_keep_order(explicit_links)
    explicit_hits = unique_keep_order(explicit_hits)

    has_any_attachment_signal = len(attachment_names) > 0
    has_supporting_doc = len(ppt_attachments) > 0 or len(doc_attachments) > 0

    if explicit_hits:
        status = "clear"
    elif has_supporting_doc or has_any_attachment_signal:
        status = "implied"
    else:
        status = "none"

    reasons = []
    if explicit_hits:
        reasons.append("explicit idea card wording found")
    if ppt_attachments:
        reasons.append("ppt/deck-like attachment detected")
    if doc_attachments:
        reasons.append("pdf/doc attachment detected")
    if has_any_attachment_signal and not (ppt_attachments or doc_attachments):
        reasons.append("attachment-like field present")
    reasons = unique_keep_order(reasons)

    return {
        "idea_card_status": status,
        "has_idea_card": status in {"clear", "implied"},
        "idea_card_clear": status == "clear",
        "idea_card_implied": status == "implied",
        "idea_card_reasons": reasons,
        "idea_card_explicit_hits": explicit_hits,
        "idea_card_links": explicit_links,
        "attachment_names": attachment_names,
        "ppt_attachments": ppt_attachments,
        "doc_attachments": doc_attachments,
        "has_ppt_attachment": len(ppt_attachments) > 0,
        "has_attachment_signal": has_any_attachment_signal,
        "text_fields_scanned": sorted(text_fields.keys()),
        "attachment_fields_scanned": sorted(attachment_fields.keys()),
    }

## 6) Aggregate ticket-level results

For each source ticket, we compute:

- `has_valid_vs` = at least one linked Theme ticket exists
- `total_vs_in_ticket` = number of distinct linked Theme tickets
- `value_stream_keys` = linked Theme keys
- `value_stream_names` = linked Theme summaries
- idea card classification and signals

In [ ]:
ticket_map = {}

for row in source_theme_rows:
    ticket = row["ticket"]
    theme = row["theme"]
    linked_group_key = row["linked_group_key"]

    if not ticket:
        continue

    ticket_key = ticket.get("key")
    if not ticket_key:
        continue

    if ticket_key not in ticket_map:
        ticket_map[ticket_key] = {
            "ticket": ticket,
            "linked_group_keys": [],
            "value_stream_keys": [],
            "value_stream_names": [],
        }

    if linked_group_key:
        ticket_map[ticket_key]["linked_group_keys"].append(linked_group_key)

    if theme:
        theme_key = theme.get("key")
        theme_summary = theme.get("summary")

        if theme_key:
            ticket_map[ticket_key]["value_stream_keys"].append(theme_key)
        if theme_summary:
            ticket_map[ticket_key]["value_stream_names"].append(theme_summary)

ticket_rows = []

for ticket_key, payload in ticket_map.items():
    ticket = payload["ticket"]
    idea = classify_idea_card(ticket)

    value_stream_keys = sorted(set(payload["value_stream_keys"]))
    value_stream_names = sorted(set(payload["value_stream_names"]))
    linked_group_keys = sorted(set(k for k in payload["linked_group_keys"] if k))

    row = {
        "ticket_key": ticket.get("key"),
        "ticket_issue_type": ticket.get("issueType"),
        "ticket_summary": ticket.get("summary"),
        "ticket_creation_date": ticket.get("creationDate"),
        "ticket_creation_epoch": ticket.get("creationDateEpoch"),
        "has_valid_vs": len(value_stream_keys) > 0,
        "total_vs_in_ticket": len(value_stream_keys),
        "linked_group_keys": linked_group_keys,
        "value_stream_keys": value_stream_keys,
        "value_stream_names": value_stream_names,
        **idea,
    }
    ticket_rows.append(row)

ticket_df = pd.DataFrame(ticket_rows).sort_values(
    by=["ticket_creation_epoch", "ticket_key"],
    ascending=[False, True],
).reset_index(drop=True)

ticket_df.head(20)

## 7) Overall stats

In [ ]:
total_tickets = len(ticket_df)
valid_ticket_df = ticket_df[ticket_df["has_valid_vs"]].copy()
invalid_ticket_df = ticket_df[~ticket_df["has_valid_vs"]].copy()

total_valid_tickets = len(valid_ticket_df)
total_invalid_tickets = len(invalid_ticket_df)

valid_with_any_idea_card = int(valid_ticket_df["has_idea_card"].sum()) if total_valid_tickets else 0
valid_with_clear_idea_card = int(valid_ticket_df["idea_card_clear"].sum()) if total_valid_tickets else 0
valid_with_implied_idea_card = int(valid_ticket_df["idea_card_implied"].sum()) if total_valid_tickets else 0
valid_with_ppt = int(valid_ticket_df["has_ppt_attachment"].sum()) if total_valid_tickets else 0

stats = {
    "since_date": SINCE_DATE,
    "source_issue_type": SOURCE_ISSUE_TYPE if SOURCE_ISSUE_TYPE else "ALL",
    "theme_issue_type": THEME_ISSUE_TYPE,
    "total_tickets_after_date": total_tickets,
    "total_valid_tickets": total_valid_tickets,
    "total_invalid_tickets": total_invalid_tickets,
    "valid_ticket_ids": valid_ticket_df["ticket_key"].tolist(),
    "valid_ticket_coverage_pct": round((total_valid_tickets / total_tickets) * 100, 2) if total_tickets else 0.0,
    "valid_tickets_with_any_idea_card": valid_with_any_idea_card,
    "valid_tickets_with_clear_idea_card": valid_with_clear_idea_card,
    "valid_tickets_with_implied_idea_card": valid_with_implied_idea_card,
    "valid_tickets_with_ppt_attachment": valid_with_ppt,
}

stats

## 8) Business-friendly views

In [ ]:
valid_ticket_df[[
    "ticket_key",
    "ticket_summary",
    "ticket_creation_date",
    "total_vs_in_ticket",
    "value_stream_names",
    "idea_card_status",
    "has_ppt_attachment",
    "attachment_names",
    "idea_card_links",
]].head(50)

In [ ]:
valid_ticket_df[valid_ticket_df["idea_card_status"] == "clear"][[
    "ticket_key",
    "ticket_summary",
    "value_stream_names",
    "idea_card_status",
    "idea_card_reasons",
    "idea_card_links",
    "attachment_names",
]].head(50)

In [ ]:
valid_ticket_df[valid_ticket_df["idea_card_status"] == "implied"][[
    "ticket_key",
    "ticket_summary",
    "value_stream_names",
    "idea_card_status",
    "idea_card_reasons",
    "has_ppt_attachment",
    "attachment_names",
]].head(50)

In [ ]:
valid_ticket_df[valid_ticket_df["idea_card_status"] == "none"][[
    "ticket_key",
    "ticket_summary",
    "value_stream_names",
    "idea_card_status",
]].head(50)

## 9) JSON output

Top-level structure:

- `summary`
- `valid_ticket_ids`
- `tickets` → one object per valid ticket

In [ ]:
valid_ticket_json_rows = []

for _, row in valid_ticket_df.iterrows():
    valid_ticket_json_rows.append({
        "ticket_id": row["ticket_key"],
        "ticket_summary": row["ticket_summary"],
        "ticket_creation_date": row["ticket_creation_date"],
        "total_value_streams": int(row["total_vs_in_ticket"]),
        "value_stream_keys": row["value_stream_keys"],
        "value_stream_names": row["value_stream_names"],
        "idea_card": {
            "has_idea_card": bool(row["has_idea_card"]),
            "status": row["idea_card_status"],
            "reasons": row["idea_card_reasons"],
            "explicit_hits": row["idea_card_explicit_hits"],
            "links": row["idea_card_links"],
            "has_ppt_attachment": bool(row["has_ppt_attachment"]),
            "attachment_names": row["attachment_names"],
            "ppt_attachments": row["ppt_attachments"],
            "doc_attachments": row["doc_attachments"],
        }
    })

final_json = {
    "summary": stats,
    "valid_ticket_ids": stats["valid_ticket_ids"],
    "tickets": valid_ticket_json_rows,
}

print(json.dumps(final_json["summary"], indent=2))

## 10) Export files

In [ ]:
ticket_df.to_csv("ticket_theme_idea_card_full_audit.csv", index=False)
valid_ticket_df.to_csv("ticket_theme_idea_card_valid_only.csv", index=False)

with open("ticket_theme_idea_card_output.json", "w", encoding="utf-8") as f:
    json.dump(final_json, f, indent=2, ensure_ascii=False)

with open("ticket_theme_idea_card_summary.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("Saved files:")
print("- ticket_theme_idea_card_full_audit.csv")
print("- ticket_theme_idea_card_valid_only.csv")
print("- ticket_theme_idea_card_output.json")
print("- ticket_theme_idea_card_summary.json")

## 11) What the output means

- `has_valid_vs` = at least one linked Theme ticket exists
- `total_vs_in_ticket` = number of distinct linked Theme tickets
- `value_stream_names` = Theme summaries used as the value streams attached
- `idea_card_status`
  - `clear` = explicit idea card wording found
  - `implied` = likely attachment-based signal only
  - `none` = nothing found
- `has_ppt_attachment` = any attachment-like field suggests PPT/PPTX/deck/slides

In [ ]:
# driver.close()